In [1]:
import time
import nest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import neo
import quantities as pq
import re
from elephant.statistics import isi, cv, mean_firing_rate
from elephant.conversion import BinnedSpikeTrain
from elephant.spike_train_correlation import corrcoef
from pathlib import Path

## import model implementation
import network
## import (default) parameters (network, simulation, stimulus)
from network_params import default_net_dict as net_dict
from sim_params import default_sim_dict as sim_dict
from stimulus_params import default_stim_dict as stim_dict

# Import library NeuroRing for FPGA and pyxrt
import neuroring
import pyxrt
from utils_binding import *   # provides .index and .bitstreamFile

# Create network and connect neurons
net = network.Network(sim_dict, net_dict, stim_dict)
net.create()
net.connect()

print(net.pops)


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.9.0
 Built: Oct  2 2025 06:57:01

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.

Data will be written to: /home/miahafiz/NeuroRing/host_py/data/
  Directory already existed. Old data will be overwritten.

Neuron numbers are scaled by a factor of 0.250.

RNG seed: 55
Total number of virtual processes: 4
Creating neuronal populations.

Mar 09 13:14:30 SimulationManager::set_status [Info]: 
    Temporal resolution changed from 0.1 to 0.1 ms.
Connecting neuronal populations recurrently.

Mar 09 13:14:45 NodeManager::prepare_nodes [Info]: 
    Preparing 19292 nodes for simulation.
[NodeCollection(metadata=None, model=iaf_psc_exp, size=5171, first=1, last=5171), NodeCollection(metadata=None, model=iaf_psc_exp, size=1458, first=5172, last=6629), No

In [2]:
param_dict = {
    'dt': 0.1,
    'tau_m': 10.0,
    'tau_syn': 0.5,
    'C_m': 250.0,
    'E_L': -65.0,
    't_ref_steps': 20,
    'V_th_abs': -50.0,
    'V_reset_abs': -65.0,
}
# 1 for recording, 0 for not recording spike
record_status = 1

host = neuroring.NeuroRingHost(net, 4096, 7000, 5, 1, param_dict, record_status, "/home/miahafiz/NeuroRing/_build_dir.hw.NUM_4096.CORE_5.FREQ_300/krnl_neuroring_hw.xclbin")


Extracting synapse information...
Loading synapse data from syndata_total19292_NperCU4096_SperCU7000.npy
Loading synapse FPGA data from synfpga_total19292_NperCU4096_SperCU7000.npy

Distributing 19292 neurons across 5 compute units on 1 FPGAs:
Kernels per FPGA: [5]

FPGA 0 (Kernels: 5):
  Kernel 0 (Global ID: 0): neurons 1 to 4096 (total: 4096)
  Kernel 1 (Global ID: 1): neurons 4097 to 8192 (total: 4096)
  Kernel 2 (Global ID: 2): neurons 8193 to 12288 (total: 4096)
  Kernel 3 (Global ID: 3): neurons 12289 to 16384 (total: 4096)
  Kernel 4 (Global ID: 4): neurons 16385 to 19292 (total: 2908)

Total neurons assigned: 19292


In [3]:
print(len(host.synapse_data))
offset = 7000
print(host.synapse_data[0*offset][0])
print(host.synapse_data[1*offset][0])


135044000
4518.0
4521.0


In [4]:
accumulated_synapse = 0
for i in range(host.total_neurons):
    accumulated_synapse += host.synapse_data[i*offset][0]
print(accumulated_synapse)

average_synapse = accumulated_synapse / host.total_neurons
print(average_synapse)

74720239.0
3873.1204126062617
